# 기억력 스포츠 성적 분석 — 2025 WMC 공식 결과 169명

**범위:** Random Words, Speed Numbers, Abstract Images, Speed Cards  
**주요 목표:** ① 데이터 품질 확인 ② Spearman 상관관계 검증 ③ K-means 전체 수준/상대 강점 패턴 비교 ④ 0점 민감도 확인

> WMSC 사무총장이 이메일로 설명된 비상업적 학부 연구 이용을 허가함.  
> 원본 선수별 기록, PDF, 재구성된 선수별 CSV를 공개 GitHub 등에 업로드하지 않습니다. 이 노트북은 **집계된 표/그래프만 내보냅니다.**

주의: 0점이 실제 실패·무효·미참가 중 무엇인지 개별 기록에서 식별되지 않습니다. 따라서 0점 포함 169명과 네 종목 모두 양수 93명의 결과를 **조건부 비교**합니다. 93명은 '결측 없는 전체 모집단'이 아니라 '네 종목 모두 양수인 선택 표본'입니다.

분석을 위해 입력 파일 `WMC_2025_four_events_integrated_169.csv`가 필요합니다. Colab에서 업로드 요청이 뜨면 해당 CSV만 선택하세요.

In [ ]:
# 환경 및 패키지
import sys, subprocess, importlib.util
for package, module in [("numpy","numpy"),("pandas","pandas"),("scipy","scipy"),
                        ("scikit-learn","sklearn"),("statsmodels","statsmodels"),
                        ("matplotlib","matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",package])

import csv, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from itertools import combinations

INPUT_FILENAME = "WMC_2025_four_events_integrated_169.csv"
candidates = [Path.cwd()/INPUT_FILENAME, Path("/mnt/data")/INPUT_FILENAME, Path("/content")/INPUT_FILENAME]
DATA_PATH = next((f for f in candidates if f.is_file()), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        print(f"분석용 CSV 파일을 업로드하세요: {INPUT_FILENAME}")
        uploaded = files.upload()
        if INPUT_FILENAME not in uploaded:
            raise FileNotFoundError(f"파일명이 {INPUT_FILENAME}인지 확인해 주세요.")
        DATA_PATH = Path.cwd()/INPUT_FILENAME
    except ImportError:
        raise FileNotFoundError(f"{INPUT_FILENAME}을 노트북과 같은 폴더에 넣으세요.")
OUT = Path("/content/WMC2025_Analysis_Outputs") if Path("/content").exists() else Path.cwd()/"WMC2025_Analysis_Outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("입력 파일:", DATA_PATH)
print("집계 출력 폴더:", OUT)

## 1. 데이터 무결성 및 0점 현황

- 분석 단위: 선수 ID 1명당 1행. 네 종목 Points 열을 사용합니다.
- 실제 0점과 미참가는 아직 구분할 수 없습니다.
- `Positive_Events`의 값도 네 종목 Points의 양수 개수와 일치하는지 검사합니다.
- 선수 이름은 분석 출력 파일에 저장하지 않습니다.

In [ ]:
COLS = {"Words":"RW_Points", "Numbers":"SN_Points",
        "Images":"AI_Points", "Cards":"SC_Final_Points"}
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
required = ["ID", *COLS.values()]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols: raise ValueError(f"필수 열이 없습니다: {missing_cols}")
if df["ID"].isna().any() or df["ID"].duplicated().any():
    raise ValueError("선수 ID가 비어 있거나 중복되었습니다.")
P = pd.DataFrame({name:pd.to_numeric(df[c],errors="coerce")
                  for name,c in COLS.items()})
if P.isna().any().any(): raise ValueError("성적 열에 빈 값 또는 변환할 수 없는 값이 있습니다.")
if (P < 0).any().any(): raise ValueError("음수 Points가 발견되었습니다. 공식 점수 확인 필요.")

positive_count = (P>0).sum(axis=1)
if "Positive_Events" in df.columns:
    assert (df["Positive_Events"].astype(int).to_numpy() == positive_count.to_numpy()).all()
all_positive = (P>0).all(axis=1)
P93 = P.loc[all_positive].reset_index(drop=True)
print("전체 선수:",len(df),"/ ID 중복: 0")
print("4종목 모두 양수:",int(all_positive.sum()))
print("4종목 모두 0:",int((positive_count==0).sum()))
print("종목별 양수 기록 수:\n", (P>0).sum().to_string())
print("양수 종목 개수별 선수 수:\n",positive_count.value_counts().sort_index().to_string())

quality = pd.DataFrame({"Discipline":list(COLS),
                        "Positive_Count":(P>0).sum().to_numpy(),
                        "Zero_Count":(P==0).sum().to_numpy()})
quality.to_csv(OUT/"data_quality_aggregate.csv",index=False,encoding="utf-8-sig")
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(quality["Discipline"],quality["Positive_Count"])
ax.set_ylim(0,len(df))
ax.set_ylabel("Athletes with positive Points")
ax.set_title("2025 WMC: Positive Scores by Discipline")
for i,v in enumerate(quality["Positive_Count"]): ax.text(i,v+2,str(v),ha="center")
fig.tight_layout()
fig.savefig(OUT/"01_positive_score_counts.png",dpi=180)
plt.show()

## 2. Spearman 상관관계 및 Holm 보정

**주 분석:** 네 종목 모두 양수인 93명에 대해 동일한 표본으로 6개 종목 쌍 비교.  
**민감도 분석 A:** 모든 169명의 0점을 그대로 포함한 결과.  
**민감도 분석 B:** 각 종목 쌍에서 두 종목 모두 양수인 선수만 비교. 이 경우 종목 쌍마다 표본 수가 달라집니다.

Spearman은 **순위**의 상관을 보기 때문에 Points를 종목별 Z-score로 바꿀 필요가 없습니다.  
`p_holm`은 여러 쌍을 검정했다는 사실을 보정한 p값입니다.

In [ ]:
def correlations(scores, mode):
    rows=[]
    for a,b in combinations(COLS.keys(),2):
        if mode=="pairwise_positive":
            valid=(scores[a]>0)&(scores[b]>0)
            a_vals,b_vals=scores.loc[valid,a],scores.loc[valid,b]
        else:
            a_vals,b_vals=scores[a],scores[b]
        rho,pval=spearmanr(a_vals,b_vals)
        rows.append({"Pair":f"{a} - {b}","n":len(a_vals),
                     "rho":float(rho),"p_raw":float(pval)})
    holm=multipletests([r["p_raw"] for r in rows],method="holm")[1]
    for r,padj in zip(rows,holm):
        r["p_holm"]=float(padj)
        r["Significant_0.05"]=bool(padj<0.05)
    return pd.DataFrame(rows)

corr93=correlations(P93,"all")
corr169=correlations(P,"all")
corrpair=correlations(P,"pairwise_positive")
corr93.assign(Sample="Four positive events").to_csv(OUT/"correlation_positive93.csv",index=False,encoding="utf-8-sig")
corr169.assign(Sample="All 169, zeros included").to_csv(OUT/"correlation_all169.csv",index=False,encoding="utf-8-sig")
corrpair.assign(Sample="Pairwise positive").to_csv(OUT/"correlation_pairwise_positive.csv",index=False,encoding="utf-8-sig")
print("주 분석(93명):")
print(corr93.to_string(index=False,formatters={"rho":lambda x:f"{x:.3f}",
                                                "p_raw":lambda x:f"{x:.2e}",
                                                "p_holm":lambda x:f"{x:.2e}"}))
print("\n전체 169명(0점 포함) rho와 비교:")
print(pd.DataFrame({"Pair":corr93["Pair"],
                    "rho_93":corr93["rho"].round(3),
                    "rho_169":corr169["rho"].round(3),
                    "rho_pair_positive":corrpair["rho"].round(3),
                    "n_pair_positive":corrpair["n"]}).to_string(index=False))

corr_mat=P93.corr(method="spearman")
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(corr_mat.values,vmin=-1,vmax=1)
ax.set_xticks(range(4),corr_mat.columns,rotation=25,ha="right")
ax.set_yticks(range(4),corr_mat.index)
for i in range(4):
    for j in range(4): ax.text(j,i,f"{corr_mat.iloc[i,j]:.2f}",ha="center",va="center")
ax.set_title("Spearman Correlations (93 Positive-Score Athletes)")
fig.colorbar(im,ax=ax,shrink=.75)
fig.tight_layout()
fig.savefig(OUT/"02_spearman_heatmap_93.png",dpi=180)
plt.show()

## 3. K-means — 세 가지 입력을 비교

1. **Points 그대로:** 숫자 크기와 분산이 다른 종목이 거리 계산에 강한 영향을 줄 수 있음
2. **종목별 Z-score:** 종목 간 척도 비교 가능. 그러나 '전체적으로 잘하는 선수'와 '못하는 선수' 구분이 남음
3. **Z-score + 선수별 평균 제거:** 각자의 평균 성적을 빼서 종목별 상대 강점 패턴에 집중

분석 대상은 모두 같은 93명으로 고정합니다. `K=2~5`와 실루엣 계수를 비교하되, 실루엣이 높아도 자동으로 진짜 '기억력 유형'이라고 판단하지 않습니다.

In [ ]:
raw=P93.to_numpy(dtype=float)
z=StandardScaler().fit_transform(raw)
profile=z-z.mean(axis=1,keepdims=True)
matrices={"Raw Points":raw,"Z-score":z,"Z-score + row centering":profile}
sil_rows=[]
models={}
for name,arr in matrices.items():
    for k in range(2,6):
        model=KMeans(n_clusters=k,random_state=42,n_init=30)
        labels=model.fit_predict(arr)
        sil=silhouette_score(arr,labels)
        sizes=np.bincount(labels,minlength=k)
        sil_rows.append({"Method":name,"K":k,"Silhouette":sil,
                         "Min_cluster_size":int(sizes.min()),
                         "Cluster_sizes":"/".join(map(str,sizes))})
        models[(name,k)]=model
sil_df=pd.DataFrame(sil_rows)
sil_df.to_csv(OUT/"cluster_silhouette_summary.csv",index=False,encoding="utf-8-sig")
print(sil_df.to_string(index=False,formatters={"Silhouette":lambda x:f"{x:.3f}"}))

fig,ax=plt.subplots(figsize=(8,4))
for name,g in sil_df.groupby("Method"):
    ax.plot(g["K"],g["Silhouette"],marker="o",label=name)
ax.set_xlabel("Number of clusters K")
ax.set_ylabel("Silhouette score")
ax.set_title("K-means Comparison (93 Positive-Score Athletes)")
ax.set_xticks([2,3,4,5])
ax.legend()
fig.tight_layout()
fig.savefig(OUT/"03_silhouette_comparison.png",dpi=180)
plt.show()

## 4. 군집의 의미 확인: '전체 실력'인지 '종목별 패턴'인지

K=2 결과를 예시로 보여줍니다. **K를 2로 선택한다고 두 가지 기억력 유형이 입증되는 것은 아닙니다.**

- 종목별 Z-score 군집의 중심이 네 종목에서 모두 + 또는 모두 - 이면 전체 경기 수준 구분일 가능성이 큽니다.
- 선수별 평균을 제거한 값의 군집 중심에서는 어떤 종목이 상대적으로 강하거나 약한지 확인할 수 있습니다.
- 실제 점수의 절대 수준도 버리지 않고 별도로 살펴봐야 합니다.

In [ ]:
names=list(COLS)
profiles=[]
for method in ["Z-score","Z-score + row centering"]:
    mat=matrices[method]
    model=models[(method,2)]
    labels=model.labels_
    means=matrices[method]
    for group in range(2):
        mask=(labels==group)
        item={"Method":method,"Cluster":group,"n":int(mask.sum()),
              "Overall_Mean_Z":float(z[mask].mean())}
        item.update({col:float(means[mask,j].mean()) for j,col in enumerate(names)})
        profiles.append(item)
profiles_df=pd.DataFrame(profiles)
profiles_df.to_csv(OUT/"cluster_profiles_aggregate.csv",index=False,encoding="utf-8-sig")
print(profiles_df.round(3).to_string(index=False))

for method,filename in [("Z-score","04_cluster_profiles_overall.png"),
                        ("Z-score + row centering","05_cluster_profiles_relative.png")]:
    g=profiles_df.loc[profiles_df["Method"]==method]
    fig,ax=plt.subplots(figsize=(8,4.5))
    for _,r in g.iterrows():
        ax.plot(names,[r[c] for c in names],marker="o",
                label=f"Group {int(r['Cluster'])} (n={int(r['n'])})")
    ax.axhline(0,linewidth=1,linestyle="--")
    ax.set_ylabel("Mean score (Z or centered Z)")
    ax.set_title(f"K=2 Cluster Profiles: {method}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT/filename,dpi=180)
    plt.show()

## 5. 민감도 점검 및 해석 한계

- 0점을 모두 포함하면 **미참가·무효·실제 실패**가 섞일 수 있고, 공통 0점이 상관계수를 높일 수 있습니다.
- 93명만 사용하면 **모든 종목에 양수 점수가 있는 선수**로 선택되는 편향이 생길 수 있습니다.
- 서로 다른 전처리를 적용했을 때 결과가 달라지는 것은 데이터 품질과 분석 목적의 문제를 드러냅니다. 민감도 분석 자체가 오염을 고치지는 않습니다.
- 초기값 반복 결과가 안정적이라고 해도 '실제 기억력 유형이 존재한다'는 증거는 아닙니다.
- 평가 조건이 다른 앱 사용자 점수를 공식 선수 군집에 직접 대입하면 안 됩니다.

In [ ]:
# 169명 전체를 단순 Z-score 군집화했을 때의 실루엣(참고용)
z169=StandardScaler().fit_transform(P.to_numpy(dtype=float))
m169=KMeans(n_clusters=2,random_state=42,n_init=30)
lab169=m169.fit_predict(z169)
s169=silhouette_score(z169,lab169)
print("93명 Z-score K=2 실루엣:",round(silhouette_score(z,models[("Z-score",2)].labels_),3))
print("169명 0점 포함 Z-score K=2 실루엣:",round(s169,3))
print("169명 군집별 수:",np.bincount(lab169).tolist())
print("주의: 169명에서 실루엣이 높아지는 것이 곧 좋은 분류라는 뜻은 아닙니다.")

# K-means 초기값 변경에 대한 참고용 안정성: 같은 93명, 같은 전처리, K=2
base=models[("Z-score + row centering",2)].labels_
ari_values=[adjusted_rand_score(base,
            KMeans(n_clusters=2,n_init=30,random_state=s).fit_predict(profile))
            for s in range(10)]
print("초기값 10개 조건의 ARI 최솟값:",round(min(ari_values),3))
print("이 수치는 '초기값 변화'에 대한 점검일 뿐, 표본 또는 전처리 안정성 보증이 아닙니다.")

summary={
 "source":"2025 World Memory Championships official discipline PDFs",
 "permission":"WMSC General Secretary confirmed described noncommercial academic use by email",
 "athletes_total":int(len(P)),
 "athletes_all_four_positive":int(len(P93)),
 "athletes_all_four_zero":int((positive_count==0).sum()),
 "event_positive_counts":{k:int((P[k]>0).sum()) for k in COLS},
 "spearman_primary":"same 93 athletes with four positive Points, six pair comparisons, Holm adjusted p",
 "cluster_primary":"Z-score across 93, then row-center for relative strengths",
 "notes":["Unknown zero reason: do not treat as confirmed absence.",
          "Four-positive-athlete subset is selected, not representative.",
          "Do not publicly redistribute athlete-level dataset or source PDFs.",
          "Correlations and clusters do not establish training benefit."]
}
(OUT/"analysis_design_notes.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")

## 6. 보고서에 쓸 수 있는 결론의 범위

**확인 가능한 결과:** 93명 조건부 표본에서 네 종목 사이의 양의 순위 상관이 관찰됨. 원점수/공식 Points와 0점 처리 조건은 기록해야 함. Z-score에서는 주로 전반적 수행 수준이 분리되고, 선수별 평균을 제거하면 종목별 상대 강점 패턴을 살펴볼 수 있음.

**아직 입증하지 못한 것:** 고유한 기억력 유형이 존재한다는 주장, 특정 훈련의 효과, 상관관계의 원인, 선수 군집에 따라 어떤 훈련이 최적인지.

**프로그램 연결:** 종목별 기록 비교 → 상대적으로 낮은 종목을 표시 → 연습 방향의 탐색적 제안 → 반복 평가. 기존 경기와 평가 조건이 다르면 선수 군집 자동 배정은 하지 않음.

**공식 데이터 출처:** WMSC 2025 네 종목 PDF.
- https://www.worldmemorychampionships.com/competitions-2025/
- https://www.worldmemorychampionships.com/wp-content/uploads/2025/12/WMC-2025-Final-Results-Random-Words.pdf
- https://www.worldmemorychampionships.com/wp-content/uploads/2025/12/WMC-2025-Final-Results-Speed-Numbers.pdf
- https://www.worldmemorychampionships.com/wp-content/uploads/2025/12/WMC-2025-Final-Results-Abstract-Images.pdf
- https://www.worldmemorychampionships.com/wp-content/uploads/2025/12/WMC-2025-Final-Results-Speed-Cards.pdf

## 6. 연령 구분(Kid / Junior / Adult / Senior) 보조 점검

WMSC 데이터에는 연령별 선수가 섞여 있습니다. 연령대별 점수 수준과 참가 패턴이 다르면 전체 상관계수와 K-means 군집에 영향을 줄 수 있습니다.

- `Category`가 있는 경우, 네 종목 모두 양수인 93명의 연령별 인원을 집계합니다.
- 상대 강점 패턴 K=2 군집의 연령별 구성을 확인합니다.
- 연령별 선수가 10명 이상인 집단에 한해 **탐색적인** Spearman 상관계수를 출력합니다.
- 연령 집단 내 표본이 작기 때문에, 여기서 나온 차이를 확정적 연령 효과로 해석하지 않습니다.

In [ ]:
if "Category" in df.columns:
    categories = df.loc[all_positive, "Category"].reset_index(drop=True)
    cat_counts = categories.value_counts(dropna=False)
    print("네 종목 모두 양수인 표본의 연령 분포:")
    print(cat_counts.to_string())

    relative_labels = models[("Z-score + row centering",2)].labels_
    age_by_cluster = pd.crosstab(
        pd.Series(relative_labels,name="Cluster"),
        categories.rename("Category"))
    print("\n상대 강점 군집별 연령 구성(집계):")
    print(age_by_cluster.to_string())
    age_by_cluster.to_csv(OUT/"age_category_by_relative_cluster_aggregate.csv",encoding="utf-8-sig")

    by_age=[]
    for cat,count in cat_counts.items():
        if count<10: continue
        subset=P93.loc[categories==cat]
        for a,b in combinations(COLS.keys(),2):
            rho,pval=spearmanr(subset[a],subset[b])
            by_age.append({"Category":cat,"n":len(subset),
                           "Pair":f"{a} - {b}","rho":float(rho)})
    age_corr=pd.DataFrame(by_age)
    age_corr.to_csv(OUT/"age_stratified_spearman_exploratory.csv",index=False,encoding="utf-8-sig")
    print("\n연령별 탐색 상관계수(집단별 n이 작으므로 참고용):")
    print(age_corr.pivot(index="Pair",columns="Category",values="rho").round(3).to_string())
else:
    print("Category 열이 없어 연령대별 보조 분석은 생략합니다.")

In [ ]:
# 공개 가능한 집계 결과만 ZIP으로 모읍니다. 선수별 데이터는 포함하지 않습니다.
zip_path=OUT.parent/"WMC2025_Aggregate_Analysis_Results.zip"
files_out=sorted(f for f in OUT.iterdir() if f.is_file())
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for f in files_out: zf.write(f,arcname=f.name)
print("생성한 결과 파일:",len(files_out),"개")
print("압축 파일:",zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("로컬 환경에서는 위 경로에서 ZIP 파일을 확인하세요.")